# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding #1 — growing versus declining content (page#6).** The paper compares pages classified as rising or falling and reports that the rising group tends to be younger and longer on average. Its outcome is the paper's up or down trend label. The methodology describes a 30-day trend comparison within a rolling 90-day study window (page#36), but does not state the exact threshold used for that paper label. These are observational group differences, not evidence that adding words or refreshing a page will definitely cause growth. Reporting the group sizes and defining the trend threshold would make the comparison easier to reproduce.

**Finding #4 — freshness and growth-to-decline ratios (page#9).** The outcome is again a growth vs.decline trend label. The paper reports a 7.88:1 ratio in the 31-90-day freshness band and explicitly warns that the 361+ band's 283:1 ratio comes from 283 growing pages and only one declining page. The important thing to know is that the tiny tail is unstable. The refreshed vs.unrefreshed health and impression differences are observational too. Pages selected for refresh may already differ. A matched comparison or a prospective refresh test would strengthen a causal claim. The paper itself says correlations do not prove causation and reports no confidence intervals or p-values (page#36).

In [5]:
import pandas as pd

paper_evidence = pd.DataFrame(
    [
        {"finding": "#1: growing vs declining", "measure": "mean word count", "group": "rising", "value": 3180, "n": 74187, "paper_page": 6},
        {"finding": "#1: growing vs declining", "measure": "mean word count", "group": "falling", "value": 2311, "n": 45272, "paper_page": 6},
        {"finding": "#1: growing vs declining", "measure": "mean content age (days)", "group": "rising", "value": 184, "n": 74187, "paper_page": 6},
        {"finding": "#1: growing vs declining", "measure": "mean content age (days)", "group": "falling", "value": 230, "n": 45272, "paper_page": 6},
        {"finding": "#4: freshness", "measure": "growth:decline ratio", "group": "31-90 days", "value": 7.88, "n": None, "paper_page": 9},
        {"finding": "#4: freshness", "measure": "growth:decline ratio", "group": "361+ days", "value": 283.0, "n": 284, "paper_page": 9},
    ]
)
display(paper_evidence)

,finding,measure,group,value,n,paper_page
0,#1: growing vs declining,mean word count,rising,3180.00,74187.0,6
1,#1: growing vs declining,mean word count,falling,2311.00,45272.0,6
2,#1: growing vs declining,mean content age (days),rising,184.00,74187.0,6
3,#1: growing vs declining,mean content age (days),falling,230.00,45272.0,6
4,#4: freshness,growth:decline ratio,31-90 days,7.88,NaN,9
5,#4: freshness,growth:decline ratio,361+ days,283.00,284.0,9


## 2. My model under an honest split (before/after)

The Week-5 notebook already uses a client holdout split. The table below compares a stratified row-wise reference with the same Random Forest evaluated on held-out clients. The row-wise score is not the honest result. The client holdout is the relevant number for transfer to unseen clients. This dataset is one snapshot, so this split does not validate possible future performance. The declining label compares recent and prior 30-day impressions, while several model features summarize the same rolling 90-day window; that overlap remains a limitation even after grouping.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "scripts" / "ml_utils.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing scripts/ml_utils.py.")

feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
if not feature_path.is_file():
    raise FileNotFoundError(
        f"Missing {feature_path}. Run `py scripts/run_all.py` from the repository root first."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from scripts.ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES

frame = pd.read_csv(feature_path)
numeric_features = [name for name in MODEL_NUMERIC_FEATURES if name in frame.columns]
categorical_features = [name for name in MODEL_CATEGORICAL_FEATURES if name in frame.columns]
final_features = numeric_features + categorical_features
required_columns = ["client_id", "is_declining_label", *final_features]
missing_columns = sorted(set(required_columns) - set(frame.columns))
if missing_columns:
    raise ValueError(f"Required columns are missing from the feature file: {missing_columns}")
if frame["client_id"].isna().any():
    raise ValueError("Client IDs are required to construct a valid grouped holdout.")

X = frame[final_features].copy()
y = frame["is_declining_label"].astype(int).reset_index(drop=True)
indices = np.arange(len(frame))

row_train_idx, row_test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)
clients = frame["client_id"].astype(str)
unique_clients = clients.unique()
rng = np.random.default_rng(42)
n_test_clients = max(1, int(round(len(unique_clients) * 0.2)))
test_clients = set(rng.permutation(unique_clients)[:n_test_clients])
group_test_mask = clients.isin(test_clients).to_numpy()
group_test_idx = indices[group_test_mask]
group_train_idx = indices[~group_test_mask]
if not set(clients.iloc[group_train_idx]).isdisjoint(set(clients.iloc[group_test_idx])):
    raise AssertionError("Client leakage: a client appears in both grouped partitions.")
if y.iloc[group_train_idx].nunique() != 2 or y.iloc[group_test_idx].nunique() != 2:
    raise ValueError("The selected client holdout must contain both label classes in train and test.")

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="constant", fill_value=0), numeric_features),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
                    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
    ]
)

def build_model():
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=10,
                    min_samples_leaf=25,
                    class_weight="balanced_subsample",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )

def precision_at_k(labels, scores, k):
    top_indices = np.argsort(np.asarray(scores))[::-1][: min(k, len(scores))]
    return float(np.asarray(labels)[top_indices].mean())

def evaluate_split(name, train_idx, test_idx):
    model = build_model()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    test_clients_count = clients.iloc[test_idx].nunique()
    return {
        "split": name,
        "test_rows": len(test_idx),
        "test_clients": test_clients_count,
        "test_base_rate": float(y.iloc[test_idx].mean()),
        "precision_at_20": precision_at_k(y.iloc[test_idx], scores, 20),
        "precision_at_50": precision_at_k(y.iloc[test_idx], scores, 50),
    }

metrics_by_split = {
    "row_random_reference": evaluate_split("row-wise reference", row_train_idx, row_test_idx),
    "client_holdout": evaluate_split("held-out clients", group_train_idx, group_test_idx),
}
split_comparison = pd.DataFrame(metrics_by_split.values())
display(split_comparison.round(3))
print(f"Overall declining base rate: {y.mean():.1%}")
print(f"Features: {len(final_features)} ({len(numeric_features)} numeric, {len(categorical_features)} categorical)")

,split,test_rows,test_clients,test_base_rate,precision_at_20,precision_at_50
0,row-wise reference,6000,31,0.542,0.9,0.90
1,held-out clients,2325,6,0.391,0.7,0.68


Overall declining base rate: 54.2%
Features: 26 (18 numeric, 8 categorical)


## 3. Leakage audit

The target is `is_declining_label = (trend_direction == 'down')`, and `trend_direction` is derived from `trend_pct`; both source columns must stay out of the model. IDs are grouping keys only, and the Week-4 rule/product flags are not model inputs. The code checks these exclusions and deliberately demonstrates the direct-label leak so the target lineage is visible. The final feature list still contains current-window traffic/engagement summaries and their tiers: because the target's 30-day trend period sits inside the 90-day snapshot, this is temporal overlap, not something a client split can fix. The available feature file has no row-level prediction dates for a clean future-time split.

In [3]:
target = frame["is_declining_label"].astype(int)
label_from_direction = frame["trend_direction"].astype(str).str.lower().eq("down").astype(int)
if not target.equals(label_from_direction):
    raise AssertionError("The stored target does not match the documented trend_direction rule.")

forbidden_features = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id",
    "baseline_score",
    "signal_declining",
    "signal_stale",
    "signal_previously_strong",
}
forbidden_present = sorted(forbidden_features.intersection(final_features))
if forbidden_present:
    raise AssertionError(f"Forbidden leakage or identifier fields are model features: {forbidden_present}")

overlapping_window_features = sorted(
    set(final_features).intersection(
        {
            "log_impressions_90d",
            "log_clicks_90d",
            "log_sessions_90d",
            "log_ai_sessions_90d",
            "days_with_impressions",
            "days_with_sessions",
            "ctr",
            "avg_position",
            "engagement_rate",
            "scroll_rate",
            "ai_traffic_pct",
            "impression_tier",
            "position_tier",
        }
    )
)

deliberate_leak_prediction = frame["trend_direction"].astype(str).str.lower().eq("down").astype(int)
leak_demo_accuracy = float((deliberate_leak_prediction == target).mean())
print(f"Target matches trend_direction == 'down': {target.equals(label_from_direction)}")
print(f"Forbidden IDs, target sources, and product flags in final features: {forbidden_present}")
print(f"Direct-leak demonstration accuracy (must not be used as a model): {leak_demo_accuracy:.1%}")
print(f"Final features with potentially overlapping snapshot windows: {overlapping_window_features}")
print("Leakage verdict: direct label columns are excluded, but temporal overlap remains unresolved.")

Target matches trend_direction == 'down': True
Forbidden IDs, target sources, and product flags in final features: []
Direct-leak demonstration accuracy (must not be used as a model): 100.0%
Final features with potentially overlapping snapshot windows: ['ai_traffic_pct', 'avg_position', 'ctr', 'days_with_impressions', 'days_with_sessions', 'engagement_rate', 'impression_tier', 'log_ai_sessions_90d', 'log_clicks_90d', 'log_impressions_90d', 'log_sessions_90d', 'position_tier', 'scroll_rate']
Leakage verdict: direct label columns are excluded, but temporal overlap remains unresolved.


## 4. Claim rewrite

A safer version of a bold model claim is: **the model ranked pages with declining labels near the top of the held-out-client list in this snapshot, at the measured precision shown below.** This is a directional result for review prioritization, not proof that refreshing those pages will definitely improve performance or that the result will hold in the future.

In [4]:
grouped_result = metrics_by_split["client_holdout"]
print(
    "Observed in this snapshot, among held-out clients, the Random Forest's top 20 contained "
    f"{grouped_result['precision_at_20']:.1%} declining-labeled pages (test base rate "
    f"{grouped_result['test_base_rate']:.1%}); its top 50 contained "
    f"{grouped_result['precision_at_50']:.1%}. This is directional decision-support for review prioritization, "
    "not evidence that a refresh causes improvement or that future-period performance will match."
)

Observed in this snapshot, among held-out clients, the Random Forest's top 20 contained 70.0% declining-labeled pages (test base rate 39.1%); its top 50 contained 68.0%. This is directional decision-support for review prioritization, not evidence that a refresh causes improvement or that future-period performance will match.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.